# ☕ Capstone Data Analysis: Coffee Shop Sales
## Tahap 3: Data Cleaning

---

**Tujuan Notebook Ini:**
- Melakukan proses Data Cleaning secara profesional berdasarkan temuan Data Quality Assessment
- Memastikan dataset memiliki kualitas yang optimal untuk analisis mendalam
- Mendokumentasikan setiap keputusan cleaning dengan justifikasi bisnis

**Dataset Input:** `data/coffee_shop_sales.csv` (dari tahap Data Quality Assessment)
**Dataset Output:** `processed/coffee_shop_sales_clean.csv`

---

# 1. Objective

Data Cleaning adalah proses mempersiapkan data mentah agar siap digunakan untuk analisis. Tahap ini sangat krusial karena **kualitas analisis bergantung pada kualitas data**. Berdasarkan Data Quality Assessment yang telah dilakukan sebelumnya, ditemukan beberapa masalah yang perlu ditangani:

1. **13 kolom memiliki tipe data yang belum optimal** (masih `object` padahal seharusnya `datetime64`, `category`, atau `bool`)
2. **4 kolom memiliki missing values** yang perlu strategi penanganan khusus
3. **Outlier** pada kolom numerik yang perlu evaluasi apakah valid atau error

Tujuan utama Data Cleaning ini adalah:
- **Memperbaiki tipe data** agar analisis temporal dan optimasi memori dapat dilakukan
- **Menangani missing values** dengan strategi yang tepat berdasarkan konteks bisnis
- **Memvalidasi outlier** dan memutuskan tindakan yang sesuai
- **Mempertahankan integritas data** - tidak menghapus data tanpa alasan yang kuat
- **Menghasilkan dataset bersih** yang siap untuk Feature Engineering dan EDA

---
# 2. Business Questions

Sebelum memulai proses cleaning, berikut adalah pertanyaan bisnis yang harus dijawab:

### Q1: Data apa saja yang perlu dibersihkan?
Berdasarkan Data Quality Assessment:
- **Tipe Data**: 13 kolom perlu konversi (1 timestamp → datetime64, 10 kolom → category, 2 kolom → bool)
- **Missing Values**: 4 kolom (holiday_name, customer_age_group, customer_gender, weather_condition)
- **Outlier**: 3 kolom numerik (quantity, unit_price, total_amount) - perlu evaluasi

### Q2: Mengapa data tersebut perlu dibersihkan?
- **Tipe data salah** menghalangi analisis temporal (misal: analisis tren bulanan)
- **Missing values** dapat menyebabkan bias dalam analisis statistik
- **Outlier** yang tidak ditangani dapat mempengaruhi perhitungan rata-rata dan model

### Q3: Bagaimana proses cleaning memengaruhi kualitas dataset?
- Tipe data yang benar → **memori lebih hemat**, query lebih cepat
- Missing values tertangani → **analisis lebih akurat**, tidak ada NaN yang mengganggu
- Outlier tervalidasi → **insight lebih reliable**, tidak ada noise dari error data

### Q4: Apakah dataset menjadi lebih layak digunakan untuk analisis?
**Ya.** Dataset akan memiliki kualitas yang lebih baik dan siap untuk:
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Statistical Modeling
- Dashboard dan Visualisasi

---
# 3. Cleaning Strategy

Berikut adalah tabel rencana cleaning berdasarkan temuan Data Quality Assessment:

| Issue | Column(s) | Action | Reason |
|-------|-----------|--------|--------|
| Wrong Data Type | `timestamp` | Konversi ke `datetime64` | Memungkinkan analisis tren waktu (harian, bulanan, tahunan) |
| Wrong Data Type | 10 kolom kategorikal | Konversi ke `category` | Menghemat memori dan mempercepat query |
| Wrong Data Type | `discount_applied`, `loyalty_member` | Konversi ke `bool` | Memudahkan filter dan kalkulasi boolean |
| Missing Value | `holiday_name` (~90%) | Biarkan NaN + buat flag `is_holiday` | Normal - hanya terisi saat hari libur |
| Missing Value | `customer_age_group` (~2-5%) | Isi dengan `'Unknown'` | Mempertahankan baris untuk analisis demografis |
| Missing Value | `customer_gender` (~2-5%) | Isi dengan `'Unknown'` | Mempertahankan baris untuk analisis demografis |
| Missing Value | `weather_condition` (~2-5%) | Isi dengan `'Unknown'` | Mempertahankan baris untuk analisis cuaca |
| Duplicate | - | Tidak ada duplikat | Dataset sudah bersih dari duplikasi |
| Invalid Value | - | Tidak ada invalid value | Tidak ditemukan quantity/price <= 0, string kosong, atau placeholder |
| Category Inconsistency | - | Tidak perlu standardisasi | Semua kategori sudah konsisten |
| Outlier | `quantity`, `unit_price`, `total_amount` | Pertahankan (valid business transactions) | Outlier merupakan transaksi valid (bulk order, pembelian premium) |

### Prinsip Cleaning:
1. **Jangan hapus data tanpa alasan kuat** - setiap baris adalah transaksi nyata
2. **Gunakan domain knowledge** - missing value pada `holiday_name` adalah normal
3. **Prioritaskan retensi data** - imputasi lebih baik daripada penghapusan
4. **Dokumentasikan setiap keputusan** - semua keputusan harus dapat dipertanggungjawabkan

---
# 4. Data Cleaning Process

Proses cleaning dilakukan secara bertahap dan terdokumentasi.

In [ ]:
# ============================================================
# Import Library
# ============================================================
import pandas as pd
import numpy as np
import os
import warnings

# Konfigurasi
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Library berhasil diimport.")

---
## 4.1 Copy Dataset

Membuat salinan dataset agar data asli tetap tersedia untuk referensi. Ini adalah best practice dalam data cleaning - **selalu kerja pada salinan, bukan data asli**.

In [ ]:
# ============================================================
# Load Dataset Asli
# ============================================================
FILE_PATH = '../data/coffee_shop_sales.csv'

df = pd.read_csv(FILE_PATH)

print(f"Dataset berhasil dimuat dari: {FILE_PATH}")
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

In [ ]:
# ============================================================
# 4.1 Copy Dataset - Buat Salinan untuk Cleaning
# ============================================================

# Simpan data asli sebagai referensi
df_original = df.copy()

# Buat salinan untuk proses cleaning
df_clean = df.copy()

print("=" * 70)
print(" 4.1 COPY DATASET")
print("=" * 70)
print(f"\nDataset asli tersimpan di: df_original")
print(f"Dataset cleaning tersimpan di: df_clean")
print(f"Shape df_clean: {df_clean.shape}")
print(f"\nPrinsip: Selalu kerja pada salinan, bukan data asli.")
print(f"Jika proses cleaning gagal, data asli masih aman.")

---
## 4.2 Fix Data Type

Berdasarkan Data Quality Assessment, terdapat **13 kolom** yang tipe datanya perlu diperbaiki:

### Alasan Perubahan:
- `timestamp` → `datetime64`: Memungkinkan ekstrak tahun, bulan, hari, jam untuk analisis tren
- 10 kolom kategorikal → `category`: Menghemat memori (hingga 90% untuk kolom dengan sedikit unique values)
- 2 kolom boolean → `bool`: Memudahkan filter `df[df['discount_applied']]` dan kalkulasi

In [ ]:
# ============================================================
# 4.2 Fix Data Type
# ============================================================

print("=" * 70)
print(" 4.2 FIX DATA TYPE")
print("=" * 70)

# Catat memori sebelum konversi
memory_before = df_clean.memory_usage(deep=True).sum() / 1024 / 1024
print(f"\nMemori SEBELUM konversi: {memory_before:.2f} MB")

# ---- Konversi timestamp ke datetime64 ----
print(f"\n[1] Konversi timestamp: object → datetime64")
print(f"    Contoh nilai: {df_clean['timestamp'].iloc[0]}")
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
print(f"    Berhasil! Tipe data: {df_clean['timestamp'].dtype}")

# ---- Konversi kolom kategorikal ke category ----
category_columns = ['city', 'country', 'store_type', 'product_category',
                    'product_name', 'payment_method', 'customer_age_group',
                    'customer_gender', 'weather_condition', 'holiday_name']

print(f"\n[2] Konversi kolom kategorikal: object → category")
for col in category_columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype('category')
        print(f"    ✓ {col}: {df_clean[col].nunique()} unique values")

# ---- Konversi kolom boolean ke bool ----
bool_columns = ['discount_applied', 'loyalty_member']

print(f"\n[3] Konversi kolom boolean: object → bool")
for col in bool_columns:
    if col in df_clean.columns:
        # Handle berbagai variasi nilai boolean
        if df_clean[col].dtype == 'object':
            mapping = {'True': True, 'False': False, 'true': True, 'false': False,
                      '1': True, '0': False, 1: True, 0: False}
            df_clean[col] = df_clean[col].map(mapping).astype('bool')
        else:
            df_clean[col] = df_clean[col].astype('bool')
        print(f"    ✓ {col}: {df_clean[col].unique()}")

# ---- Hitung memori setelah konversi ----
memory_after = df_clean.memory_usage(deep=True).sum() / 1024 / 1024
memory_saved = memory_before - memory_after
memory_pct = (memory_saved / memory_before * 100)

print(f"\n{'='*50}")
print(f"Memori SETELAH konversi: {memory_after:.2f} MB")
print(f"Penghematan memori: {memory_saved:.2f} MB ({memory_pct:.1f}%)")

# Verifikasi tipe data
print(f"\n[VERIFIKASI] Tipe Data Setelah Konversi:")
dtype_summary = pd.DataFrame({
    'Column': df_clean.columns,
    'New Type': [str(df_clean[col].dtype) for col in df_clean.columns]
})
print(dtype_summary.to_string(index=False))

### Hasil Fix Data Type:

| Kolom | Sebelum | Sesudah | Penghematan |
|-------|---------|---------|-------------|
| timestamp | object | datetime64 | Memungkinkan analisis temporal |
| 10 kolom kategorikal | object | category | Hemat memori ~90% per kolom |
| 2 kolom boolean | object | bool | Memudahkan filter dan kalkulasi |

**Total penghematan memori** terlihat pada output di atas. Konversi ke `category` sangat efektif untuk kolom dengan sedikit unique values (seperti `country` dengan 4 unique values dari 20,000 baris).

---
## 4.3 Handle Missing Value

Berdasarkan Data Quality Assessment, terdapat **4 kolom** dengan missing values. Strategi penanganan disesuaikan dengan konteks bisnis masing-masing kolom.

| Kolom | Missing | Severity | Strategi | Alasan Bisnis |
|-------|---------|----------|----------|---------------|
| `holiday_name` | ~90% | Low | Biarkan NaN + buat flag `is_holiday` | Normal - hanya terisi saat hari libur |
| `customer_age_group` | ~2-5% | Medium | Isi `'Unknown'` | Mempertahankan baris untuk analisis |
| `customer_gender` | ~2-5% | Medium | Isi `'Unknown'` | Mempertahankan baris untuk analisis |
| `weather_condition` | ~2-5% | Medium | Isi `'Unknown'` | Mempertahankan baris untuk analisis |

In [ ]:
# ============================================================
# 4.3 Handle Missing Value
# ============================================================

print("=" * 70)
print(" 4.3 HANDLE MISSING VALUE")
print("=" * 70)

# Hitung missing values SEBELUM cleaning
missing_before = df_clean.isnull().sum()
missing_before_pct = (df_clean.isnull().sum() / len(df_clean) * 100).round(2)

print(f"\nMISSING VALUES SEBELUM CLEANING:")
print("-" * 50)
for col in df_clean.columns:
    if missing_before[col] > 0:
        print(f"  {col}: {missing_before[col]:,} ({missing_before_pct[col]:.1f}%)")

total_missing_before = df_clean.isnull().sum().sum()
print(f"\nTotal missing values: {total_missing_before:,}")

In [ ]:
# ---- 4.3.1 holiday_name: Biarkan NaN + buat flag is_holiday ----
print("\n[1] holiday_name (~90% missing)")
print("    Strategi: Biarkan NaN, buat kolom flag 'is_holiday'")
print("    Alasan: Missing value NORMAL - hanya terisi saat hari libur.")
print("    Tidak perlu imputasi karena NaN = 'bukan hari libur'.")

# Buat kolom flag is_holiday
df_clean['is_holiday'] = df_clean['holiday_name'].notna().astype('bool')

print(f"    Kolom 'is_holiday' dibuat.")
print(f"    Distribusi: {df_clean['is_holiday'].value_counts().to_dict()}")
print(f"    Holiday transactions: {df_clean['is_holiday'].sum():,} ({df_clean['is_holiday'].mean()*100:.1f}%)")

# holiday_name tetap NaN untuk baris non-holiday (tidak diimputasi)

In [ ]:
# ---- 4.3.2 customer_age_group: Isi dengan 'Unknown' ----
print("\n[2] customer_age_group")
missing_age = df_clean['customer_age_group'].isnull().sum()
print(f"    Missing: {missing_age:,} ({missing_age/len(df_clean)*100:.1f}%)")
print("    Strategi: Isi dengan 'Unknown'")
print("    Alasan: Data demografis tidak lengkap, imputasi dengan 'Unknown'")
print("    mempertahankan baris untuk analisis segmentasi pelanggan.")

df_clean['customer_age_group'] = df_clean['customer_age_group'].cat.add_categories(['Unknown'])
df_clean['customer_age_group'] = df_clean['customer_age_group'].fillna('Unknown')

missing_age_after = df_clean['customer_age_group'].isnull().sum()
print(f"    Missing SETELAH: {missing_age_after:,}")
print(f"    Distribusi: {df_clean['customer_age_group'].value_counts().to_dict()}")

In [ ]:
# ---- 4.3.3 customer_gender: Isi dengan 'Unknown' ----
print("\n[3] customer_gender")
missing_gender = df_clean['customer_gender'].isnull().sum()
print(f"    Missing: {missing_gender:,} ({missing_gender/len(df_clean)*100:.1f}%)")
print("    Strategi: Isi dengan 'Unknown'")
print("    Alasan: Pelanggan tidak memberikan info gender, imputasi 'Unknown'")
print("    menjaga integritas data tanpa mengarang informasi.")

df_clean['customer_gender'] = df_clean['customer_gender'].cat.add_categories(['Unknown'])
df_clean['customer_gender'] = df_clean['customer_gender'].fillna('Unknown')

missing_gender_after = df_clean['customer_gender'].isnull().sum()
print(f"    Missing SETELAH: {missing_gender_after:,}")
print(f"    Distribusi: {df_clean['customer_gender'].value_counts().to_dict()}")

In [ ]:
# ---- 4.3.4 weather_condition: Isi dengan 'Unknown' ----
print("\n[4] weather_condition")
missing_weather = df_clean['weather_condition'].isnull().sum()
print(f"    Missing: {missing_weather:,} ({missing_weather/len(df_clean)*100:.1f}%)")
print("    Strategi: Isi dengan 'Unknown'")
print("    Alasan: Data cuaca tidak tersedia untuk semua lokasi/waktu.")
print("    'Unknown' merepresentasikan ketidakpastian data, bukan nilai yang salah.")

df_clean['weather_condition'] = df_clean['weather_condition'].cat.add_categories(['Unknown'])
df_clean['weather_condition'] = df_clean['weather_condition'].fillna('Unknown')

missing_weather_after = df_clean['weather_condition'].isnull().sum()
print(f"    Missing SETELAH: {missing_weather_after:,}")
print(f"    Distribusi: {df_clean['weather_condition'].value_counts().to_dict()}")

In [ ]:
# ---- Ringkasan Missing Values SETELAH cleaning ----
print("\n" + "=" * 70)
print(" RINGKASAN MISSING VALUES")
print("=" * 70)

missing_after = df_clean.isnull().sum()
missing_after_pct = (df_clean.isnull().sum() / len(df_clean) * 100).round(2)

summary = pd.DataFrame({
    'Column': df_clean.columns,
    'Missing Before': missing_before,
    'Missing After': missing_after,
    'Status': ['Fixed' if missing_before[col] > 0 and missing_after[col] == 0
               else 'Intentional (NaN retained)' if col == 'holiday_name' and missing_after[col] > 0
               else 'No Action Needed' for col in df_clean.columns]
})

summary = summary[summary['Missing Before'] > 0]
print("\n" + summary.to_string(index=False))

total_missing_after = df_clean.isnull().sum().sum()
print(f"\nTotal missing BEFORE: {total_missing_before:,}")
print(f"Total missing AFTER : {total_missing_after:,}")
print(f"Missing yang diatasi: {total_missing_before - total_missing_after:,}")
print(f"\nCatatan: holiday_name sengaja dipertahankan NaN karena itu adalah")
print(f"representasi yang benar dari data (bukan hari libur).")

### Hasil Handle Missing Value:

| Kolom | Missing Before | Missing After | Status |
|-------|----------------|---------------|--------|
| `holiday_name` | ~18,000+ | ~18,000+ | **Intentional** - NaN = bukan hari libur |
| `customer_age_group` | ~400-1000 | **0** | **Fixed** - diisi 'Unknown' |
| `customer_gender` | ~400-1000 | **0** | **Fixed** - diisi 'Unknown' |
| `weather_condition` | ~400-1000 | **0** | **Fixed** - diisi 'Unknown' |

**Prinsip yang diikuti:**
- `holiday_name`: NaN adalah representasi yang benar (bukan hari libur)
- Kolom lain: Imputasi 'Unknown' lebih baik daripada menghapus baris

---
## 4.4 Remove Duplicate

Berdasarkan Data Quality Assessment, dataset memiliki **0 duplicate rows**. Namun, kita tetap melakukan pengecekan ulang untuk memastikan.

In [ ]:
# ============================================================
# 4.4 Remove Duplicate
# ============================================================

print("=" * 70)
print(" 4.4 REMOVE DUPLICATE")
print("=" * 70)

# Pengecekan duplicate SEBELUM
duplicate_before = df_clean.duplicated().sum()
print(f"\nDuplicate rows SEBELUM: {duplicate_before:,}")

if duplicate_before > 0:
    print(f"\nMenampilkan contoh duplicate:")
    print(df_clean[df_clean.duplicated(keep='first')].head())
    
    # Hapus duplicate
    df_clean = df_clean.drop_duplicates(keep='first').reset_index(drop=True)
    
    duplicate_after = df_clean.duplicated().sum()
    print(f"\nDuplicate rows SETELAH: {duplicate_after:,}")
    print(f"Baris dihapus: {duplicate_before - duplicate_after:,}")
    print(f"Alasan bisnis: Duplikasi data transaksi akan mengakibatkan")
    print(f"overcounting dalam analisis pendapatan dan volume penjualan.")
else:
    print(f"\nStatus: TIDAK ADA DUPLIKASI ✓")
    print(f"Dataset sudah bersih dari duplikasi.")
    print(f"Alasan: Setiap baris merepresentasikan transaksi unik.")
    print(f"transaction_id sudah terverifikasi unik dari Data Quality Assessment.")

print(f"\nShape dataset: {df_clean.shape}")

---
## 4.5 Fix Invalid Value

Memperbaiki nilai yang tidak valid secara logis atau bisnis:
- Quantity <= 0
- Harga <= 0
- String kosong
- Nilai placeholder (Unknown, N/A, dll)
- Spasi berlebih

Berdasarkan Data Quality Assessment, **tidak ditemukan invalid values**. Namun, kita tetap melakukan pengecekan ulang.

In [ ]:
# ============================================================
# 4.5 Fix Invalid Value
# ============================================================

print("=" * 70)
print(" 4.5 FIX INVALID VALUE")
print("=" * 70)

invalid_issues = []

# ---- 1. Quantity <= 0 ----
if 'quantity' in df_clean.columns:
    invalid_qty = (df_clean['quantity'] <= 0).sum()
    print(f"\n[1] Quantity <= 0: {invalid_qty:,}")
    if invalid_qty > 0:
        print(f"    Status: INVALID - Quantity harus > 0")
        # Hapus baris dengan quantity <= 0
        df_clean = df_clean[df_clean['quantity'] > 0].reset_index(drop=True)
        print(f"    Action: Baris dihapus")
        invalid_issues.append({'Check': 'quantity <= 0', 'Count': invalid_qty, 'Action': 'Dihapus'})
    else:
        print(f"    Status: VALID ✓")

# ---- 2. Unit Price <= 0 ----
if 'unit_price' in df_clean.columns:
    invalid_price = (df_clean['unit_price'] <= 0).sum()
    print(f"\n[2] Unit Price <= 0: {invalid_price:,}")
    if invalid_price > 0:
        print(f"    Status: INVALID - Harga harus > 0")
        df_clean = df_clean[df_clean['unit_price'] > 0].reset_index(drop=True)
        print(f"    Action: Baris dihapus")
        invalid_issues.append({'Check': 'unit_price <= 0', 'Count': invalid_price, 'Action': 'Dihapus'})
    else:
        print(f"    Status: VALID ✓")

# ---- 3. Total Amount <= 0 ----
if 'total_amount' in df_clean.columns:
    invalid_total = (df_clean['total_amount'] <= 0).sum()
    print(f"\n[3] Total Amount <= 0: {invalid_total:,}")
    if invalid_total > 0:
        print(f"    Status: INVALID - Total harus > 0")
        df_clean = df_clean[df_clean['total_amount'] > 0].reset_index(drop=True)
        print(f"    Action: Baris dihapus")
        invalid_issues.append({'Check': 'total_amount <= 0', 'Count': invalid_total, 'Action': 'Dihapus'})
    else:
        print(f"    Status: VALID ✓")

# ---- 4. String kosong ----
print(f"\n[4] String Kosong (empty string)")
string_cols = df_clean.select_dtypes(include=['object', 'category']).columns
empty_found = False
for col in string_cols:
    if df_clean[col].dtype == 'category':
        empty_count = (df_clean[col].astype(str).str.strip() == '').sum()
    else:
        empty_count = (df_clean[col].str.strip() == '').sum()
    if empty_count > 0:
        print(f"    {col}: {empty_count:,} string kosong")
        empty_found = True
        invalid_issues.append({'Check': f'Empty string in {col}', 'Count': empty_count, 'Action': 'Dihapus'})
if not empty_found:
    print(f"    Status: TIDAK ADA STRING KOSONG ✓")

# ---- 5. Nilai placeholder ----
print(f"\n[5] Nilai Placeholder")
placeholder_values = ['Unknown', 'N/A', 'NA', 'null', 'NULL', '-', 'none', 'NONE']
placeholder_found = False
for col in string_cols:
    for placeholder in placeholder_values:
        if df_clean[col].dtype == 'category':
            count = (df_clean[col].astype(str).str.lower() == placeholder.lower()).sum()
        else:
            count = (df_clean[col].str.lower() == placeholder.lower()).sum()
        if count > 0:
            print(f"    {col}: '{placeholder}' sebanyak {count:,}")
            placeholder_found = True
if not placeholder_found:
    print(f"    Status: TIDAK ADA PLACEHOLDER ✓")

# ---- 6. Spasi berlebih ----
print(f"\n[6] Spasi Berlebih")
space_found = False
for col in string_cols:
    if df_clean[col].notna().any():
        if df_clean[col].dtype == 'category':
            has_leading = (df_clean[col].astype(str).str.strip() != df_clean[col].astype(str)).sum()
        else:
            has_leading = (df_clean[col].str.strip() != df_clean[col]).sum()
        if has_leading > 0:
            print(f"    {col}: {has_leading:,} spasi berlebih")
            space_found = True
            # Bersihkan spasi
            if df_clean[col].dtype == 'category':
                df_clean[col] = df_clean[col].astype(str).str.strip().astype('category')
            else:
                df_clean[col] = df_clean[col].str.strip()
            print(f"    Action: Spasi dibersihkan")
if not space_found:
    print(f"    Status: TIDAK ADA SPASI BERLEBIH ✓")

# ---- Ringkasan ----
print(f"\n{'='*50}")
print(f"RINGKASAN INVALID VALUES")
print(f"{'='*50}")
if invalid_issues:
    for issue in invalid_issues:
        print(f"  {issue['Check']}: {issue['Count']:,} baris - {issue['Action']}")
else:
    print(f"  TIDAK ADA INVALID VALUE DITEMUKAN ✓")

print(f"\nShape dataset: {df_clean.shape}")

---
## 4.6 Standardize Category

Memastikan semua kategori memiliki penulisan yang konsisten:
- Hilangkan spasi berlebih (leading/trailing)
- Standardisasi kapitalisasi
- Pastikan tidak ada kategori ganda karena typo

Berdasarkan Data Quality Assessment, **semua kategori sudah konsisten**. Namun, kita tetap melakukan standarisasi untuk memastikan.

In [ ]:
# ============================================================
# 4.6 Standardize Category
# ============================================================

print("=" * 70)
print(" 4.6 STANDARDIZE CATEGORY")
print("=" * 70)

category_columns = ['city', 'country', 'store_type', 'product_category',
                    'product_name', 'payment_method', 'customer_age_group',
                    'customer_gender', 'weather_condition', 'holiday_name']

changes_made = []

for col in category_columns:
    if col in df_clean.columns:
        original_categories = set(df_clean[col].cat.categories)
        
        # Strip whitespace dari semua kategori
        new_categories = [cat.strip() if isinstance(cat, str) else cat for cat in original_categories]
        
        # Standardisasi kapitalisasi (Title Case untuk semua)
        new_categories = [cat.title() if isinstance(cat, str) else cat for cat in new_categories]
        
        # Cek apakah ada perubahan
        if set(new_categories) != original_categories:
            # Mapping perubahan
            mapping = {old: new for old, new in zip(original_categories, new_categories) if old != new}
            
            # Apply mapping
            for old, new in mapping.items():
                df_clean[col] = df_clean[col].cat.rename_categories({old: new})
            
            changes_made.append({'Column': col, 'Changes': len(mapping)})
            print(f"\n  {col}:")
            print(f"    Perubahan: {mapping}")
        else:
            print(f"  {col}: Konsisten ✓")

# Ringkasan
print(f"\n{'='*50}")
print(f"RINGKASAN STANDARDISASI")
print(f"{'='*50}")
if changes_made:
    for change in changes_made:
        print(f"  {change['Column']}: {change['Changes']} kategori diperbaiki")
else:
    print(f"  Semua kategori sudah konsisten. Tidak ada perubahan diperlukan.")

# Tampilkan unique values untuk verifikasi
print(f"\n[VERIFIKASI] Unique Values Setelah Standardisasi:")
for col in category_columns:
    if col in df_clean.columns:
        unique_vals = sorted([str(v) for v in df_clean[col].cat.categories if str(v) != 'nan'])
        print(f"  {col}: {unique_vals}")

---
## 4.7 Handle Outlier

Berdasarkan Data Quality Assessment, outlier ditemukan pada kolom `quantity`, `unit_price`, dan `total_amount`. Namun, **outlier tersebut merupakan transaksi bisnis yang valid**:
- **quantity tinggi**: Bulk order (pesanan catering atau event)
- **unit_price tinggi**: Produk premium atau spesial
- **total_amount tinggi**: Pembelian banyak item atau kombinasi produk premium

### Keputusan: **PERTAHANKAN SEMUA OUTLIER**

Alasan bisnis:
1. Outlier adalah transaksi nyata yang terjadi
2. Menghapus outlier akan menghilangkan informasi penting tentang perilaku pembelian
3. Bulk order dan pembelian premium adalah bagian valid dari bisnis coffee shop
4. Untuk analisis statistik, gunakan median (robust terhadap outlier) bukan mean

In [ ]:
# ============================================================
# 4.7 Handle Outlier
# ============================================================

print("=" * 70)
print(" 4.7 HANDLE OUTLIER")
print("=" * 70)

# Kolom numerik untuk analisis outlier
numeric_cols = ['unit_price', 'quantity', 'temperature_c', 'total_amount']

outlier_summary = []

for col in numeric_cols:
    if col in df_clean.columns:
        data = df_clean[col].dropna()
        
        # IQR Method
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = data[(data < lower_bound) | (data > upper_bound)]
        outlier_count = len(outliers)
        outlier_pct = (outlier_count / len(data) * 100).round(2)
        
        # Tentukan keputusan
        if col == 'temperature_c':
            decision = 'Pertahankan - Suhu ekstrem adalah data valid'
        elif col == 'quantity':
            decision = 'Pertahankan - Bulk order adalah transaksi valid'
        elif col == 'unit_price':
            decision = 'Pertahankan - Harga premium adalah data valid'
        elif col == 'total_amount':
            decision = 'Pertahankan - Transaksi besar adalah data valid'
        
        outlier_summary.append({
            'Column': col,
            'Q1': Q1,
            'Q3': Q3,
            'IQR': IQR,
            'Lower Bound': lower_bound,
            'Upper Bound': upper_bound,
            'Outlier Count': outlier_count,
            'Outlier %': outlier_pct,
            'Decision': 'Pertahankan'
        })
        
        print(f"\n{col.upper()}:")
        print(f"  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
        print(f"  Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
        print(f"  Outliers: {outlier_count:,} ({outlier_pct}%)")
        print(f"  Keputusan: PERTAHANKAN")
        print(f"  Alasan: {decision}")

# Ringkasan
print(f"\n{'='*70}")
print(f"RINGKASAN OUTLIER")
print(f"{'='*70}")
outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df[['Column', 'Outlier Count', 'Outlier %', 'Decision']].to_string(index=False))

print(f"\nKESIMPULAN:")
print(f"Semua outlier DIPERTAHANKAN karena merupakan transaksi bisnis yang valid.")
print(f"Outlier bukan error data, melainkan representasi dari:")
print(f"  - Bulk order (quantity tinggi)")
print(f"  - Produk premium (unit_price tinggi)")
print(f"  - Transaksi besar (total_amount tinggi)")
print(f"  - Cuaca ekstrem (temperature_c ekstrem)")
print(f"\nUntuk analisis statistik, gunakan MEDIAN (robust terhadap outlier)")
print(f"bukan MEAN (sensitive terhadap outlier).")

---
## 4.8 Final Validation

Memastikan seluruh proses cleaning berhasil dan dataset siap untuk analisis.

In [ ]:
# ============================================================
# 4.8 Final Validation
# ============================================================

print("=" * 70)
print(" 4.8 FINAL VALIDATION")
print("=" * 70)

validation_results = []

# ---- 1. Missing Values ----
missing_remaining = df_clean.isnull().sum().sum()
missing_cols = [col for col in df_clean.columns if df_clean[col].isnull().any()]
print(f"\n[1] MISSING VALUES")
print(f"    Total missing values: {missing_remaining:,}")
print(f"    Kolom dengan missing: {missing_cols}")
if missing_cols == ['holiday_name']:
    print(f"    Status: VALID - holiday_name sengaja dipertahankan NaN ✓")
    validation_results.append({'Check': 'Missing Values', 'Status': '✓ Pass', 'Detail': 'Only holiday_name (intentional)'})
elif len(missing_cols) == 0:
    print(f"    Status: VALID - Tidak ada missing values ✓")
    validation_results.append({'Check': 'Missing Values', 'Status': '✓ Pass', 'Detail': 'No missing values'})
else:
    print(f"    Status: WARNING - Masih ada missing values yang perlu ditangani ⚠")
    validation_results.append({'Check': 'Missing Values', 'Status': '⚠ Warning', 'Detail': f'{missing_cols}'})

# ---- 2. Duplicate ----
duplicate_count = df_clean.duplicated().sum()
print(f"\n[2] DUPLICATE ROWS")
print(f"    Jumlah duplicate: {duplicate_count:,}")
if duplicate_count == 0:
    print(f"    Status: VALID - Tidak ada duplikasi ✓")
    validation_results.append({'Check': 'Duplicates', 'Status': '✓ Pass', 'Detail': '0 duplicates'})
else:
    print(f"    Status: FAIL - Masih ada duplikasi ⚠")
    validation_results.append({'Check': 'Duplicates', 'Status': '✗ Fail', 'Detail': f'{duplicate_count} duplicates'})

# ---- 3. Data Types ----
print(f"\n[3] DATA TYPES")
expected_types = {
    'timestamp': 'datetime64[ns]',
    'discount_applied': 'bool',
    'loyalty_member': 'bool',
    'is_holiday': 'bool'
}
type_correct = True
for col, expected in expected_types.items():
    if col in df_clean.columns:
        actual = str(df_clean[col].dtype)
        if actual != expected:
            print(f"    {col}: {actual} (expected {expected}) ✗")
            type_correct = False
        else:
            print(f"    {col}: {actual} ✓")

# Cek kategorikal
cat_cols = ['city', 'country', 'store_type', 'product_category',
            'product_name', 'payment_method', 'customer_age_group',
            'customer_gender', 'weather_condition', 'holiday_name']
for col in cat_cols:
    if col in df_clean.columns:
        if df_clean[col].dtype.name == 'category':
            pass  # OK
        else:
            print(f"    {col}: {df_clean[col].dtype} (expected category) ✗")
            type_correct = False

if type_correct:
    print(f"    Status: VALID - Semua tipe data benar ✓")
    validation_results.append({'Check': 'Data Types', 'Status': '✓ Pass', 'Detail': 'All types correct'})
else:
    print(f"    Status: WARNING - Beberapa tipe data perlu diperbaiki ⚠")
    validation_results.append({'Check': 'Data Types', 'Status': '⚠ Warning', 'Detail': 'Some types incorrect'})

# ---- 4. Category Consistency ----
print(f"\n[4] CATEGORY CONSISTENCY")
cat_issues = 0
for col in cat_cols:
    if col in df_clean.columns:
        # Cek spasi berlebih
        categories = df_clean[col].cat.categories
        for cat in categories:
            if isinstance(cat, str):
                if cat != cat.strip():
                    cat_issues += 1
                    print(f"    {col}: '{cat}' memiliki spasi berlebih")
                if cat != cat.title():
                    cat_issues += 1
                    print(f"    {col}: '{cat}' perlu standardisasi kapitalisasi")

if cat_issues == 0:
    print(f"    Status: VALID - Semua kategori konsisten ✓")
    validation_results.append({'Check': 'Category Consistency', 'Status': '✓ Pass', 'Detail': 'All consistent'})
else:
    print(f"    Status: WARNING - {cat_issues} masalah ditemukan ⚠")
    validation_results.append({'Check': 'Category Consistency', 'Status': '⚠ Warning', 'Detail': f'{cat_issues} issues'})

# ---- 5. Dataset Ready ----
print(f"\n[5] DATASET READINESS")
print(f"    Shape: {df_clean.shape}")
print(f"    Memory: {df_clean.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
print(f"    Columns: {list(df_clean.columns)}")

# ---- Ringkasan Validasi ----
print(f"\n{'='*70}")
print(f"RINGKASAN VALIDASI AKHIR")
print(f"{'='*70}")
validation_df = pd.DataFrame(validation_results)
print(validation_df.to_string(index=False))

# Cek apakah semua pass
all_pass = all('Pass' in status for status in validation_df['Status'])
if all_pass:
    print(f"\n✓ SEMUA VALIDASI PAS - Dataset siap untuk analisis!")
else:
    print(f"\n⚠ BEBERAPA VALIDASI PERLU PERHATIAN - Review kembali proses cleaning.")

---
# 5. Before vs After Cleaning

Perbandingan kondisi dataset sebelum dan sesudah proses cleaning.

In [ ]:
# ============================================================
# 5. Before vs After Cleaning
# ============================================================

print("=" * 70)
print(" 5. BEFORE vs AFTER CLEANING")
print("=" * 70)

# Hitung metrik BEFORE
rows_before = df_original.shape[0]
cols_before = df_original.shape[1]
missing_before_total = df_original.isnull().sum().sum()
dup_before = df_original.duplicated().sum()
cat_before = sum(1 for col in df_original.columns if df_original[col].dtype == 'object')

# Hitung metrik AFTER
rows_after = df_clean.shape[0]
cols_after = df_clean.shape[1]
missing_after_total = df_clean.isnull().sum().sum()
dup_after = df_clean.duplicated().sum()
cat_after = sum(1 for col in df_clean.columns if df_clean[col].dtype.name == 'category')

# Outlier (tetap sama karena dipertahankan)
outlier_count = 0
for col in ['unit_price', 'quantity', 'temperature_c', 'total_amount']:
    if col in df_clean.columns:
        data = df_clean[col].dropna()
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        outlier_count += ((data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)).sum()

# Buat tabel perbandingan
comparison = pd.DataFrame({
    'Metric': ['Jumlah Baris', 'Jumlah Kolom', 'Missing Values', 'Duplicate Rows',
               'Kolom Kategorikal', 'Kolom Boolean', 'Kolom Datetime', 'Outlier (total)'],
    'Before': [
        f"{rows_before:,}",
        f"{cols_before}",
        f"{missing_before_total:,}",
        f"{dup_before}",
        f"{sum(1 for col in df_original.columns if df_original[col].dtype == 'object')}",
        f"{sum(1 for col in df_original.columns if df_original[col].dtype == 'bool')}",
        f"{sum(1 for col in df_original.columns if 'datetime' in str(df_original[col].dtype))}",
        f"~{outlier_count:,}"
    ],
    'After': [
        f"{rows_after:,}",
        f"{cols_after} (+1: is_holiday)",
        f"{missing_after_total:,} (holiday_name)",
        f"{dup_after}",
        f"{cat_after}",
        f"{sum(1 for col in df_clean.columns if df_clean[col].dtype == 'bool')}",
        f"{sum(1 for col in df_clean.columns if 'datetime' in str(df_clean[col].dtype))}",
        f"~{outlier_count:,} (dipertahankan)"
    ],
    'Change': [
        f"{'Same' if rows_before == rows_after else str(rows_after - rows_before)}",
        f"+1 (is_holiday)",
        f"-{missing_before_total - missing_after_total}",
        f"{'Same' if dup_before == dup_after else str(dup_after - dup_before)}",
        f"object → category",
        f"object → bool",
        f"object → datetime64",
        f"Dipertahankan"
    ]
})

print("\n" + comparison.to_string(index=False))

print(f"\n{'='*70}")
print(f"KESIMPULAN PERBANDINGAN")
print(f"{'='*70}")
print(f"1. Baris: {'Tidak berubah (tidak ada data dihapus)' if rows_before == rows_after else f'Berkurang {rows_before - rows_after} baris'}")
print(f"2. Kolom: Bertambah 1 (kolom flag 'is_holiday')")
print(f"3. Missing Values: Berkurang dari {missing_before_total:,} ke {missing_after_total:,}")
print(f"4. Duplicate: {'Tidak ada perubahan (sudah 0)' if dup_before == dup_after else 'Berkurang'}")
print(f"5. Tipe Data: 13 kolom sudah dikonversi ke tipe yang benar")
print(f"6. Outlier: Dipertahankan (merupakan transaksi valid)")

---
# 6. Cleaning Log

Dokumentasi seluruh proses cleaning yang telah dilakukan.

In [ ]:
# ============================================================
# 6. Cleaning Log
# ============================================================

print("=" * 90)
print(" 6. CLEANING LOG - DOKUMENTASI PROSES")
print("=" * 90)

cleaning_log = pd.DataFrame({
    'No': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    'Cleaning Action': [
        'Copy Dataset',
        'Konversi Tipe Data',
        'Konversi Tipe Data',
        'Konversi Tipe Data',
        'Handle Missing Value',
        'Handle Missing Value',
        'Handle Missing Value',
        'Handle Missing Value',
        'Remove Duplicate',
        'Fix Invalid Value',
        'Standardize Category',
        'Handle Outlier',
        'Create Flag Column',
        'Final Validation',
        'Export Dataset'
    ],
    'Column(s)': [
        'All',
        'timestamp',
        '10 kolom kategorikal',
        'discount_applied, loyalty_member',
        'holiday_name',
        'customer_age_group',
        'customer_gender',
        'weather_condition',
        'All',
        'quantity, unit_price, total_amount',
        'Semua kolom kategorikal',
        'quantity, unit_price, total_amount, temperature_c',
        'holiday_name → is_holiday',
        'All',
        'coffee_shop_sales_clean.csv'
    ],
    'Before': [
        f'{rows_before:,} rows × {cols_before} cols',
        'object',
        'object',
        'object',
        f'~{df_original["holiday_name"].isnull().sum():,} missing',
        f'{df_original["customer_age_group"].isnull().sum():,} missing',
        f'{df_original["customer_gender"].isnull().sum():,} missing',
        f'{df_original["weather_condition"].isnull().sum():,} missing',
        f'{dup_before} duplicates',
        'Tidak ada invalid',
        'Sudah konsisten',
        'Outlier ditemukan',
        'Tidak ada',
        'N/A',
        'N/A'
    ],
    'After': [
        f'{rows_after:,} rows × {cols_after} cols',
        'datetime64[ns]',
        'category',
        'bool',
        'NaN dipertahankan',
        '0 missing (Unknown)',
        '0 missing (Unknown)',
        '0 missing (Unknown)',
        '0 duplicates',
        'Tidak ada invalid',
        'Tetap konsisten',
        'Dipertahankan',
        'is_holiday (bool)',
        'All checks passed',
        'processed/coffee_shop_sales_clean.csv'
    ],
    'Reason': [
        'Data asli tetap tersedia sebagai referensi',
        'Memungkinkan analisis tren waktu',
        'Menghemat memori dan mempercepat query',
        'Memudahkan filter dan kalkulasi boolean',
        'Normal - hanya terisi saat hari libur',
        'Mempertahankan baris untuk analisis',
        'Mempertahankan baris untuk analisis',
        'Mempertahankan baris untuk analisis',
        'Dataset sudah bersih dari duplikasi',
        'Dataset tidak memiliki invalid values',
        'Memastikan konsistensi kategori',
        'Outlier merupakan transaksi valid',
        'Memudahkan analisis hari libur',
        'Memastikan kualitas data akhir',
        'Menyimpan hasil cleaning'
    ]
})

print("\n" + cleaning_log.to_string(index=False))

print(f"\nTotal langkah cleaning: {len(cleaning_log)}")
print(f"File output: processed/coffee_shop_sales_clean.csv")

---
# 7. Findings

Temuan selama proses Data Cleaning:

In [ ]:
# ============================================================
# 7. Findings
# ============================================================

print("=" * 80)
print(" 7. FINDINGS: TEMUAN SELAMA PROSES CLEANING")
print("=" * 80)

findings = """
TEMUAN 1: KONVERSI TIPE DATA MEMBERIKAN PENGHEMATAN MEMORI SIGNIFIKAN
=======================================================================
Konversi 10 kolom kategorikal dari 'object' ke 'category' memberikan
penghematan memori yang signifikan. Untuk kolom seperti 'country' dengan
hanya 4 unique values dari 20,000 baris, penghematan bisa mencapai 90%.
Konversi timestamp ke datetime64 memungkinkan analisis tren temporal
(harian, mingguan, bulanan) yang sebelumnya tidak mungkin dilakukan.

TEMUAN 2: MISSING VALUE PADA HOLIDAY_NAME ADALAH NORMAL
========================================================
Missing value ~90% pada kolom 'holiday_name' bukanlah masalah kualitas data,
melainkan representasi yang benar dari realitas bisnis. Kolom ini hanya terisi
saat transaksi terjadi pada hari libur. Dibuat kolom flag 'is_holiday' untuk
memudahkan analisis tanpa kehilangan informasi.

TEMUAN 3: MISSING VALUE PADA KOLOM DEMOGRAFIS BISA DIIMPUTASI
==============================================================
Missing value ~2-5% pada customer_age_group, customer_gender, dan
weather_condition berhasil ditangani dengan imputasi 'Unknown'.
Pendekatan ini lebih baik daripada menghapus baris karena:
- Mempertahankan 100% data transaksi
- Tidak mengarang informasi (menggunakan 'Unknown' sebagai placeholder)
- Memungkinkan analisis tetap berjalan untuk kolom lain

TEMUAN 4: DATASET SUDAH SANGAT BERSIH DARI AWAL
=================================================
Data Quality Assessment menunjukkan dataset sudah dalam kondisi sangat baik:
- 0 duplicate rows
- 0 invalid values (quantity/price <= 0, string kosong, placeholder)
- Kategori sudah konsisten (tidak ada typo atau inkonsistensi)
Hal ini menunjukkan proses data collection dan entry sudah dilakukan dengan baik.

TEMUAN 5: OUTLIER MERUPAKAN TRANSAKSI BISNIS YANG VALID
=========================================================
Outlier yang ditemukan pada quantity, unit_price, dan total_amount
merupakan representasi dari:
- Bulk order (quantity > 10): Pesanan catering atau event
- Harga premium (unit_price tinggi): Produk spesial atau limited edition
- Transaksi besar (total_amount tinggi): Pembelian banyak item sekaligus
Menghapus outlier akan menghilangkan informasi berharga tentang
perilaku pembelian pelanggan.

TEMUAN 6: PROSES CLEANING TIDAK MENGHILANGKAN DATA
====================================================
Jumlah baris tetap 20,000 setelah cleaning. Tidak ada data yang dihapus
karena tidak ditemukan alasan yang cukup kuat untuk penghapusan.
Semua keputusan cleaning berfokus pada RETENSI data, bukan penghapusan.
"""

print(findings)

---
# 8. Conclusion

Jawaban atas seluruh Business Questions yang diajukan di awal:

In [ ]:
# ============================================================
# 8. Conclusion
# ============================================================

print("=" * 80)
print(" 8. CONCLUSION: JAWABAN BUSINESS QUESTIONS")
print("=" * 80)

conclusion = """
Q1: Data apa saja yang dibersihkan?
A1: 
    1. TIPE DATA: 13 kolom dikonversi
       - timestamp: object → datetime64
       - 10 kolom kategorikal: object → category
       - 2 kolom boolean: object → bool
    
    2. MISSING VALUES: 4 kolom ditangani
       - holiday_name: NaN dipertahankan + flag is_holiday dibuat
       - customer_age_group: diisi 'Unknown'
       - customer_gender: diisi 'Unknown'
       - weather_condition: diisi 'Unknown'
    
    3. OUTLIER: Dievaluasi, semua dipertahankan
       - quantity, unit_price, total_amount, temperature_c
    
    4. CATEGORY: Distandardisasi (spasi & kapitalisasi)


Q2: Mengapa data tersebut dibersihkan?
A2:
    - Tipe data salah menghalangi analisis temporal dan optimasi memori
    - Missing values dapat menyebabkan bias dalam analisis
    - Outlier perlu dievaluasi apakah error atau data valid
    - Standardisasi memastikan konsistensi untuk reporting


Q3: Bagaimana cleaning meningkatkan kualitas data?
A3:
    BEFORE → AFTER:
    - Memori: Lebih hemat setelah konversi ke category/datetime
    - Missing values: Berkualitas lebih baik (tidak ada imputasi sembarangan)
    - Tipe data: Semua kolom memiliki tipe yang benar
    - Kategori: Konsisten dan terstandardisasi
    - Flag baru: is_holiday memudahkan analisis hari libur


Q4: Apakah dataset sudah siap untuk Feature Engineering?
A4: YA - Dataset sudah sangat siap untuk Feature Engineering:
    - Timestamp sudah datetime → bisa ekstrak tahun, bulan, hari, jam
    - Kategori sudah category → bisa dilakukan one-hot encoding
    - Boolean sudah bool → bisa langsung digunakan untuk filter
    - Missing values tertangani → tidak ada NaN yang mengganggu
    - is_holiday sudah tersedia → siap untuk analisis hari libur
    - Outlier dipertahankan → data tetap lengkap untuk analisis
"""

print(conclusion)

---
# 9. Export Clean Dataset

Menyimpan dataset hasil cleaning ke folder `processed/`.

In [ ]:
# ============================================================
# 9. Export Clean Dataset
# ============================================================

print("=" * 70)
print(" 9. EXPORT CLEAN DATASET")
print("=" * 70)

# Buat folder processed jika belum ada
PROCESSED_DIR = '../processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"\nFolder '{PROCESSED_DIR}' siap.")

# Simpan dataset bersih
EXPORT_PATH = os.path.join(PROCESSED_DIR, 'coffee_shop_sales_clean.csv')
df_clean.to_csv(EXPORT_PATH, index=False)

file_size = os.path.getsize(EXPORT_PATH) / 1024

print(f"\nDataset berhasil disimpan ke: {EXPORT_PATH}")
print(f"Ukuran file: {file_size:.2f} KB")
print(f"Jumlah baris: {len(df_clean):,}")
print(f"Jumlah kolom: {len(df_clean.columns)}")

# Verifikasi
print(f"\n{'='*50}")
print(f"VERIFIKASI EXPORT")
print(f"{'='*50}")

df_verify = pd.read_csv(EXPORT_PATH)

print(f"\nBerhasil dibaca kembali: {EXPORT_PATH}")
print(f"Shape: {df_verify.shape}")
print(f"Kolom: {list(df_verify.columns)}")
print(f"\n5 baris pertama:")
df_verify.head()

In [ ]:
# ============================================================
# Akhir Notebook - Data Cleaning
# ============================================================

print("\n" + "*" * 80)
print("*", " " * 28, "DATA CLEANING SELESAI", " " * 29, "*")
print("*" * 80)
print(f"\nDataset: processed/coffee_shop_sales_clean.csv")
print(f"Shape: {df_clean.shape}")
print(f"Status: DATASET SIAP UNTUK FEATURE ENGINEERING & EDA")
print(f"\nTimestamp: {pd.Timestamp.now()}")
print("*" * 80)